# Imports

In [1]:
from train import train, TrainingConfig
import os
from helpers import load_bpe_tokenization, load_encoding, save_model
import torch
from datasets import StrideDataset
from simpleGPT import SimpleGPTConfig, SimpleGPT
from block import BlockConfig

# Path configuration

In [2]:
models_dir_name = models_dir_path = "models"
model_name = "mickiewicz_gpt_010"
tokenizers_dir_name = tokenizers_dir_path = "tokenizers"
tokenizer_name = "tokenizer_mickiewicz_010"
encodings_dir_name = encodings_dir_path = "encodings"
tr_encoding_name = "tr_encoding_mickiewicz_010"
val_encoding_name = "val_encoding_mickiewicz_010"

# Tokenizer

In [3]:
encode, decode, vocab, merges = load_bpe_tokenization(os.path.join(tokenizers_dir_path, tokenizer_name + ".pt"))

# Encodings

In [4]:
tr_encoding = load_encoding(os.path.join(encodings_dir_path, tr_encoding_name + ".pt"))
val_encoding = load_encoding(os.path.join(encodings_dir_path, val_encoding_name + ".pt"))

# Datasets

In [5]:
BLOCK_SIZE = 128
tr_dataset = StrideDataset(tr_encoding, BLOCK_SIZE, stride=BLOCK_SIZE//8)
val_dataset = StrideDataset(val_encoding, BLOCK_SIZE, stride=BLOCK_SIZE//8)

# Model

In [6]:
N_BLOCKS = 8
N_EMBD = 384
N_HEADS = 6
ATTENTION_INNER_DIM = 64
FF_EMBD_TO_DIM_RATIO = 4.0
DROPOUT = 0.1
model = SimpleGPT(SimpleGPTConfig(
    vocab_size=len(vocab),
    n_blocks=N_BLOCKS,
    block_config=BlockConfig(
        n_heads=N_HEADS,
        n_embd=N_EMBD,
        attention_inner_dim=ATTENTION_INNER_DIM,
        ff_embedding_to_dim_ratio=FF_EMBD_TO_DIM_RATIO,
        dropout=DROPOUT
    )
))
print(f"Model has: {sum([p.numel() for p in model.parameters()])} learnable parameters")

Model has: 17324028 learnable parameters


# Training

In [7]:
BATCH_SIZE = 192
NUM_EPOCHS = 15
LR = 1e-4
INFO_INTERVAL = 1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
training_config = TrainingConfig(
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    lr=LR,
    info_interval=INFO_INTERVAL,
    device=DEVICE
)
train(model, tr_dataset, val_dataset, training_config)


Using device: cuda
Train dataset length: 20154 | Val dataset length: 2232
Epoch 0 | Train loss: 8.5612 | Val loss: 8.5558
Epoch 1 | Train loss: 7.6887 | Val loss: 7.5518
Epoch 2 | Train loss: 7.5622 | Val loss: 7.5464
Epoch 3 | Train loss: 7.5427 | Val loss: 7.5038
Epoch 4 | Train loss: 7.1184 | Val loss: 6.8352
Epoch 5 | Train loss: 6.5786 | Val loss: 6.5524
Epoch 6 | Train loss: 6.2814 | Val loss: 6.3533
Epoch 7 | Train loss: 6.0402 | Val loss: 6.2165
Epoch 8 | Train loss: 5.8345 | Val loss: 6.1699
Epoch 9 | Train loss: 5.6718 | Val loss: 6.0538
Epoch 10 | Train loss: 5.5364 | Val loss: 6.0077
Epoch 11 | Train loss: 5.4217 | Val loss: 5.9678
Epoch 12 | Train loss: 5.3221 | Val loss: 5.9594
Epoch 13 | Train loss: 5.2350 | Val loss: 5.9355
Epoch 14 | Train loss: 5.1504 | Val loss: 5.9209
Epoch 15 | Train loss: 5.0674 | Val loss: 5.9165


# Saving checkpoint

In [8]:
model.to('cpu')
save_model(model, model.config, os.path.join(models_dir_path, model_name + ".pt"))

[save_model] Model saved to models/mickiewicz_gpt_010.pt


# Further training

In [9]:
INFO_INTERVAL = 5
NUM_EPOCHS = 50
training_config.info_interval = INFO_INTERVAL
training_config.num_epochs = NUM_EPOCHS
train(model, tr_dataset, val_dataset, training_config)

Using device: cuda
Train dataset length: 20154 | Val dataset length: 2232
Epoch 0 | Train loss: 4.8937 | Val loss: 5.9165
Epoch 5 | Train loss: 4.7480 | Val loss: 5.9108
Epoch 10 | Train loss: 4.3767 | Val loss: 5.9586
Epoch 15 | Train loss: 3.9699 | Val loss: 6.1118
Epoch 20 | Train loss: 3.5254 | Val loss: 6.3527
Epoch 25 | Train loss: 3.0649 | Val loss: 6.6619
Epoch 30 | Train loss: 2.6115 | Val loss: 7.0250
Epoch 35 | Train loss: 2.1882 | Val loss: 7.4258
Epoch 40 | Train loss: 1.8167 | Val loss: 7.8545
Epoch 45 | Train loss: 1.5013 | Val loss: 8.2820
Epoch 50 | Train loss: 1.2383 | Val loss: 8.7084
